<a href="https://colab.research.google.com/github/KarlaMichelleSorianoSanhez/Simulacion-I/blob/main/M%C3%A9todo_para_determinar_cuando_detenerse_al_generar_nuevos_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad: Método para determinar cuándo detenerse al generar nuevos datos

Estimar mediante simulación Monte Carlo la integral

$$
I=\int_0^1 e^{x^2}\,dx.
$$

Para ello, generar observaciones de la variable aleatoria adecuada y utilizar el método estudiado para determinar cuándo detener la generación de nuevos datos.

El criterio de paro consiste en continuar simulando hasta que

$$
\frac{S}{\sqrt{k}}<d,
$$

donde:

- $S$ es la desviación estándar muestral.
- $k$ es el número de observaciones generadas.
- $d$ es una tolerancia previamente establecida.

Como condición inicial, generar al menos 100 observaciones antes de comenzar a verificar el criterio de paro.

Realizar el procedimiento para los siguientes valores de la tolerancia:

$$
d=0.1
$$

y

$$
d=0.001.
$$

Finalmente, reportar para cada caso:

- La estimación de la integral.
- La desviación estándar muestral.
- El error estándar estimado

$$
\frac{S}{\sqrt{k}}.
$$

- El número total de observaciones requeridas para satisfacer el criterio de paro.

Además, comparar los resultados obtenidos y analizar cómo influye la tolerancia elegida en la cantidad de observaciones necesarias para alcanzar la precisión solicitada.

### Objetivo

El objetivo de esta actividad es implementar el método propuesto para determinar cuándo detener la generación de nuevos datos en una simulación Monte Carlo.

La idea consiste en generar observaciones de una variable aleatoria cuya esperanza coincida con la cantidad que se desea estimar y continuar generando datos hasta que la precisión de la estimación sea suficientemente buena.

Para medir dicha precisión se utilizará la desviación estándar estimada de la media muestral.

### Construcción del estimador Monte Carlo

Sea

$$
I=\int_a^b g(x)\,dx.
$$

Si

$$
U\sim U(a,b),
$$

entonces la densidad de $U$ es

$$
f_U(u)=\frac{1}{b-a},
\qquad a\leq u\leq b.
$$

Por definición de esperanza,

$$
E[g(U)]
=
\int_a^b g(u)\frac{1}{b-a}\,du.
$$

Multiplicando ambos lados por $(b-a)$ se obtiene

$$
(b-a)E[g(U)]
=
\int_a^b g(u)\,du.
$$

Por lo tanto,

$$
I=E[X],
$$

donde

$$
X=(b-a)g(U).
$$

Así, la integral puede estimarse mediante la media muestral de observaciones de la variable aleatoria $X$.

### Media muestral

Si se generan observaciones independientes

$$
X_1,X_2,\ldots,X_n,
$$

la media muestral se define como

$$
\bar X_n
=
\frac{1}{n}
\sum_{i=1}^{n}X_i.
$$

De acuerdo con la teoría, la media muestral es un estimador insesgado de

$$
E[X].
$$

Por esta razón utilizaremos $\bar X_n$ como estimador de la integral.

### Varianza muestral

Para medir la variabilidad de las observaciones se utiliza la varianza muestral

$$
S_n^2
=
\frac{
\sum_{i=1}^{n}
(X_i-\bar X_n)^2
}{n-1}.
$$

La desviación estándar muestral es

$$
S_n=\sqrt{S_n^2}.
$$

Además, la desviación estándar estimada de la media muestral es

$$
\frac{S_n}{\sqrt n}.
$$

Esta cantidad mide la precisión de la estimación obtenida.

### Criterio de paro

El método estudiado propone continuar generando observaciones hasta que

$$
\frac{S_n}{\sqrt n}<d,
$$

donde $d$ representa una tolerancia elegida previamente.

Mientras esta condición no se cumpla, se siguen generando nuevas observaciones.

En esta actividad se utilizarán los valores

$$
d=0.1
$$

y

$$
d=0.001.
$$

Además, por indicación de la actividad, se generarán inicialmente al menos 100 observaciones antes de comenzar a verificar el criterio de paro.

In [ ]:
import numpy as np

### Función a integrar

La integral que se desea estimar es

$$
I=\int_0^1 e^{x^2}\,dx.
$$

Por lo tanto, la función que interviene en el estimador Monte Carlo es

$$
g(x)=e^{x^2}.
$$

Se define una función independiente para facilitar la reutilización del algoritmo.

In [ ]:
# Función del problema

def g(x):
    return np.exp(x**2)

### Generación de una observación Monte Carlo

Si

$$
U\sim U(a,b),
$$

entonces

$$
I=\int_a^b g(x)\,dx
=
E[(b-a)g(U)].
$$

Por esta razón, cada observación simulada será

$$
X=(b-a)g(U).
$$

Se crea una función específica para generar una sola observación.

In [ ]:
# Generar una observación Monte Carlo

def generar_observacion(g, a, b):

    u = np.random.uniform(a, b)

    x = (b - a) * g(u)

    return x

### Actualización recursiva de la media

De acuerdo con las notas, la media muestral puede actualizarse mediante

$$
\bar X_{j+1}
=
\bar X_j
+
\frac{X_{j+1}-\bar X_j}{j+1}.
$$

Esta fórmula evita recalcular toda la suma cada vez que se genera una nueva observación.

In [ ]:
# Actualizar media muestral

def actualizar_media(media, x, k):

    nueva_media = media + (x - media)/(k + 1)

    return nueva_media

### Actualización recursiva de la varianza

Las notas presentan la relación

$$
S_{j+1}^{2}
=
\left(1-\frac1j\right)S_j^2
+
(j+1)
(\bar X_{j+1}-\bar X_j)^2.
$$

Esta expresión permite actualizar la varianza muestral utilizando únicamente la información disponible en el paso anterior.

In [ ]:
# Actualizar varianza muestral

def actualizar_varianza(S2, media_anterior, media_nueva, k):

    nueva_S2 = (
        (1 - 1/k)*S2
        +
        (k + 1)*(media_nueva - media_anterior)**2
    )

    return nueva_S2

### Criterio de paro

El algoritmo continúa generando observaciones mientras

$$
\frac{S}{\sqrt{k}}
\geq d.
$$

Cuando se cumple

$$
\frac{S}{\sqrt{k}}
<
d,
$$

se considera que la precisión requerida ha sido alcanzada y la simulación se detiene.

In [ ]:
# Verificar criterio de paro

def criterio_paro(S2, k, d):

    error_estandar = np.sqrt(S2) / np.sqrt(k)

    return error_estandar < d

### Algoritmo principal

Se implementa el procedimiento completo descrito en las notas.

Primero se generan al menos 100 observaciones.

Posteriormente se sigue simulando hasta que se satisfaga el criterio

$$
\frac{S}{\sqrt{k}}<d.
$$

Finalmente se reportan:

$$
\bar X,
\qquad
S,
\qquad
\frac{S}{\sqrt{k}},
\qquad
k.
$$

In [ ]:
# ==========================================================
# Simulación Monte Carlo con criterio de paro
# ==========================================================

def monte_carlo_paro(g, a, b, d, n_min=100):

    # Primera observación
    x = generar_observacion(g, a, b)

    media = x
    S2 = 0.0
    k = 1

    # --------------------------------------------------
    # Observaciones iniciales
    # --------------------------------------------------

    while k < n_min:

        x = generar_observacion(g, a, b)

        media_anterior = media

        media = actualizar_media(media, x, k)

        if k >= 1:
            S2 = actualizar_varianza(
                S2,
                media_anterior,
                media,
                k
            )

        k += 1

    # --------------------------------------------------
    # Criterio de paro
    # --------------------------------------------------

    while not criterio_paro(S2, k, d):

        x = generar_observacion(g, a, b)

        media_anterior = media

        media = actualizar_media(media, x, k)

        S2 = actualizar_varianza(
            S2,
            media_anterior,
            media,
            k
        )

        k += 1

    S = np.sqrt(S2)

    error = S / np.sqrt(k)

    return media, S, error, k

### Caso 1

Se ejecuta el algoritmo utilizando

$$
d=0.1.
$$

Este valor permite obtener una estimación con una precisión moderada.

In [ ]:
# ==========================================================
# Caso d = 0.01
# ==========================================================

media_01, S_01, error_01, n_01 = monte_carlo_paro(
    g,
    0,
    1,
    0.01
)

print("RESULTADOS PARA d = 0.01")
print(f"Estimación = {media_01:.8f}")
print(f"Desviación estándar = {S_01:.8f}")
print(f"Error estándar = {error_01:.8f}")
print(f"Número de observaciones = {n_01}")

RESULTADOS PARA d = 0.01
Estimación = 1.45604739
Desviación estándar = 0.46720546
Error estándar = 0.00999956
Número de observaciones = 2183


### Caso 2

Se repite el procedimiento utilizando

$$
d=0.001.
$$

Al exigir una precisión mucho mayor, el algoritmo requerirá generar una cantidad considerablemente más grande de observaciones.

In [ ]:
# ==========================================================
# Caso d = 0.001
# ==========================================================

media_001, S_001, error_001, n_001 = monte_carlo_paro(
    g,
    0,
    1,
    0.001
)

print("RESULTADOS PARA d = 0.001")
print(f"Estimación = {media_001:.8f}")
print(f"Desviación estándar = {S_001:.8f}")
print(f"Error estándar = {error_001:.8f}")
print(f"Número de observaciones = {n_001}")

RESULTADOS PARA d = 0.001
Estimación = 1.46268610
Desviación estándar = 0.47520510
Error estándar = 0.00100000
Número de observaciones = 225821


### Comparación final

Finalmente se comparan ambos experimentos para observar cómo influye la tolerancia seleccionada en el número de observaciones requeridas.

Mientras más pequeño sea el valor de

$$
d,
$$

mayor será el esfuerzo computacional necesario para alcanzar la precisión solicitada.

In [ ]:
# ==========================================================
# Comparación final
# ==========================================================

print("\nCOMPARACIÓN FINAL")
print("-"*70)

print(
    f"{'d':<10}"
    f"{'Estimación':<20}"
    f"{'Error estándar':<20}"
    f"{'Observaciones':<15}"
)

print("-"*70)

print(
    f"{0.01:<10}"
    f"{media_01:<20.8f}"
    f"{error_01:<20.8f}"
    f"{n_01:<15}"
)

print(
    f"{0.001:<10}"
    f"{media_001:<20.8f}"
    f"{error_001:<20.8f}"
    f"{n_001:<15}"
)


COMPARACIÓN FINAL
----------------------------------------------------------------------
d         Estimación          Error estándar      Observaciones  
----------------------------------------------------------------------
0.01      1.45604739          0.00999956          2183           
0.001     1.46268610          0.00100000          225821         
